In [ ]:
# Music For Robots (M4R) — Experiment D (Albot Einstein): "Finish the Human's Phrase"
# Gives MusicGen (facebook/musicgen-small) the first half of 4 real unquantized Michael Luchtan recordings
# (ties-that-bide, bided-ties, gnossienne-01, rusty-electrics) and asks the AI to finish the phrase!
# Then compares the AI continuation vs. the real human ending on:
#   - Spectral Centroid (Hz), Dynamic Range (dB), Onset Density (onsets/s), Inter-Onset Interval Irregularity (IOI CV = rubato vs quantization!),
#   - MFCC Cosine Similarity (timbral match), and >16 kHz High-Frequency Air!
# Saves playable WAV files (prompt + human ending vs. prompt + AI ending) so humans & robots can hear the difference directly.

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "librosa", "soundfile", "scipy", "tabulate"])

import json
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration

DATA_ROOT = Path("/tmp/m4r-data")
if not (DATA_ROOT / "tracks").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/luchtan-GIT/m4r-data.git", str(DATA_ROOT)])

OUT_AUDIO = Path("/kaggle/working/continuations")
OUT_AUDIO.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running MusicGen continuation on:", device)

MODEL_ID = "facebook/musicgen-small"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = MusicgenForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
model.eval()
SR = model.config.audio_encoder.sampling_rate  # 32000 Hz
print("MusicGen sampling rate:", SR)

TRACKS = [
    ("ties-that-bide", 4.5, 4.75, "unquantized solo acoustic miniature, sparse resonant notes, raw room decay"),
    ("bided-ties", 10.0, 10.0, "unquantized acoustic miniature, hesitant rubato phrasing, intimate room tone"),
    ("gnossienne-01", 12.0, 12.0, "bar-line-free solo piano, Erik Satie Gnossienne, rubato hesitation, expressive pauses"),
    ("rusty-electrics", 12.0, 12.0, "unquantized electric guitar, raw corroded tone, sudden silence and microtiming drift"),
    ("cathartic-callbacks", 12.0, 12.0, "unquantized solo acoustic guitar, wide dynamic range, human finger hesitation"),
]

def acoustic_features(y, sr):
    y = y.astype(np.float32)
    if len(y) < 1024:
        return {}
    rms = librosa.feature.rms(y=y)[0]
    rms_mean = float(np.mean(rms))
    rms_max = float(np.max(rms))
    rms_p10 = float(np.percentile(rms[rms > 1e-6], 10)) if np.any(rms > 1e-6) else 1e-6
    dr_db = float(20.0 * np.log10(max(rms_max, 1e-6) / max(rms_p10, 1e-6)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)[0]))
    onsets = librosa.onset.onset_detect(y=y, sr=sr, units="time", delta=0.10, wait=6)
    dur = len(y) / float(sr)
    onset_rate = len(onsets) / max(dur, 0.1)
    if len(onsets) >= 3:
        ioi = np.diff(onsets)
        ioi_cv = float(np.std(ioi) / max(np.mean(ioi), 1e-6))
    else:
        ioi_cv = 0.0
    mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)
    # High frequency energy ratio (>8kHz for 32kHz SR)
    S = np.abs(librosa.stft(y)) ** 2
    freqs = librosa.fft_frequencies(sr=sr)
    hi = float(np.sum(S[freqs >= 8000]))
    lo = float(np.sum(S[freqs < 8000])) + 1e-12
    hi_ratio = hi / lo
    return {
        "centroid_hz": round(centroid, 1),
        "dr_db": round(dr_db, 1),
        "rms_mean": round(rms_mean, 5),
        "onset_rate_per_s": round(onset_rate, 2),
        "ioi_cv_rubato": round(ioi_cv, 3),
        "hi_freq_ratio": round(hi_ratio, 5),
        "mfcc": mfcc,
    }

def cos_sim(a, b):
    return float(np.dot(a, b) / (max(np.linalg.norm(a), 1e-9) * max(np.linalg.norm(b), 1e-9)))

rows = []
for tid, prompt_sec, cont_sec, text_hint in TRACKS:
    mp3_path = DATA_ROOT / "tracks" / tid / "audio.mp3"
    y_full, _ = librosa.load(str(mp3_path), sr=SR, mono=True)
    p_len = int(prompt_sec * SR)
    c_len = int(cont_sec * SR)
    y_prompt = y_full[:p_len]
    y_human_cont = y_full[p_len:p_len + c_len]

    # Save real human prompt + continuation
    sf.write(str(OUT_AUDIO / f"{tid}_human_full.wav"), np.concatenate([y_prompt, y_human_cont]), SR)

    # Run 2 modes: (1) Blind audio-only continuation (no text hint), (2) Text-guided audio continuation
    for mode_name, prompt_text in [("AudioOnly_Blind", ""), ("AudioPlusTextHint", text_hint)]:
        torch.manual_seed(42)
        inputs = processor(
            audio=y_prompt,
            sampling_rate=SR,
            text=[prompt_text],
            padding=True,
            return_tensors="pt",
        ).to(device)
        # MusicGen generates at 50 tokens/sec
        total_tokens = int((prompt_sec + cont_sec) * 50)
        with torch.no_grad():
            audio_values = model.generate(**inputs, do_sample=True, guidance_scale=3.0, max_new_tokens=max(int(cont_sec * 50), 150))
        y_gen_full = audio_values[0, 0].cpu().numpy()
        # Extract the continuation portion after prompt_sec
        if len(y_gen_full) > p_len + int(1.0 * SR):
            y_ai_cont = y_gen_full[p_len:p_len + c_len]
        else:
            y_ai_cont = y_gen_full[:c_len]

        sf.write(str(OUT_AUDIO / f"{tid}_ai_{mode_name}.wav"), y_gen_full, SR)

        fh = acoustic_features(y_human_cont, SR)
        fa = acoustic_features(y_ai_cont, SR)
        sim = cos_sim(fh["mfcc"], fa["mfcc"])

        row = {
            "track_id": tid,
            "mode": mode_name,
            "prompt_sec": prompt_sec,
            "cont_sec": cont_sec,
            "mfcc_timbre_similarity": round(sim, 4),
            "human_centroid_hz": fh["centroid_hz"],
            "ai_centroid_hz": fa["centroid_hz"],
            "centroid_err_hz": round(abs(fa["centroid_hz"] - fh["centroid_hz"]), 1),
            "human_dr_db": fh["dr_db"],
            "ai_dr_db": fa["dr_db"],
            "dr_diff_db": round(fa["dr_db"] - fh["dr_db"], 1),
            "human_onset_rate": fh["onset_rate_per_s"],
            "ai_onset_rate": fa["onset_rate_per_s"],
            "human_rubato_ioi_cv": fh["ioi_cv_rubato"],
            "ai_rubato_ioi_cv": fa["ioi_cv_rubato"],
            "rubato_retention_ratio": round(fa["ioi_cv_rubato"] / max(fh["ioi_cv_rubato"], 0.01), 3),
        }
        rows.append(row)
        print(f"[{tid} | {mode_name}] MFCC_sim={sim:.3f} | Human DR={fh['dr_db']}dB vs AI DR={fa['dr_db']}dB | Human Rubato CV={fh['ioi_cv_rubato']} vs AI CV={fa['ioi_cv_rubato']}")

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/experiment_d_results.csv", index=False)

summary = [
    "# Experiment D Summary — 'Finish the Human's Phrase' (MusicGen Audio Continuation vs. Real Human Ending)\n",
    df[["track_id", "mode", "mfcc_timbre_similarity", "human_dr_db", "ai_dr_db", "human_rubato_ioi_cv", "ai_rubato_ioi_cv", "human_onset_rate", "ai_onset_rate"]].to_markdown(index=False),
    "\n\n## Mean Comparison Across Modes\n",
    df.groupby("mode")[["mfcc_timbre_similarity", "centroid_err_hz", "dr_diff_db", "human_rubato_ioi_cv", "ai_rubato_ioi_cv"]].mean().reset_index().to_markdown(index=False),
]
Path("/kaggle/working/experiment_d_summary.md").write_text("\n".join(summary))
print("\n".join(summary))
